# Language Identification on IndicCorpV2 (from scratch, student style)

**Goal:** build a language identifier for 23 Indian language(+script) labels using
IndicCorpV2 sentences.

**Rules we follow (as per the assignment):**
1. Take 1000 sentences per language from IndicCorpV2 (paragraphs are split into
   sentences ourselves).
2. Split 80 / 10 / 10 into train / val / test, **stratified** (same counts per
   language in each split).
3. Represent each sentence with **TF-IDF** built using:
   - word unigrams + word bigrams
   - character 2-grams, 3-grams, 4-grams
   - **no `sklearn`** — the vectorizer is written by hand.
4. Classify with a **multinomial (softmax) Logistic Regression** written by hand
   (no `sklearn.linear_model`), trained with plain gradient descent.
5. Evaluate with **macro-F1**, computed by hand (no `sklearn.metrics`).

Everything here is written in a very plain, "student" way — simple loops,
simple dictionaries, simple numpy — so every step is easy to follow. It is
*not* optimized for speed, so be patient when it runs.


In [ ]:
!pip install -q datasets huggingface_hub pandas numpy


In [ ]:
import re
import json
import random
import math
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from datasets import load_dataset

random.seed(42)
np.random.seed(42)


## Step 1: Build the raw dataset (1000 sentences per language)

The dataset repo's own split/config resolution (`split="tel_Telu"` etc.)
turned out to be unreliable across different versions of the `datasets`
library — it sometimes falls back to looking for a folder that doesn't
exist and throws `EmptyDatasetError`.

The **robust fix** is to skip that machinery entirely and stream the raw
`.txt` files straight from the repo's `data/` folder using the generic
`"text"` loader (every line in the file becomes one example). This does
not depend on any repo-specific script/config logic, so it just works.

23 languages, ISO-ish short codes used as our class labels:


In [ ]:
# 23 language labels (short codes) -> human readable name (with script)
LANGS = {
    "as":  "Assamese  (Bengali-Assamese)",
    "bd":  "Bodo      (Devanagari)",
    "bn":  "Bengali   (Bengali)",
    "dg":  "Dogri     (Devanagari)",
    "gom": "Konkani   (Devanagari)",
    "gu":  "Gujarati  (Gujarati)",
    "hi":  "Hindi     (Devanagari)",
    "kha": "Khasi     (Latin)",
    "kn":  "Kannada   (Kannada)",
    "ks":  "Kashmiri  (Perso-Arabic)",
    "mai": "Maithili  (Devanagari)",
    "ml":  "Malayalam (Malayalam)",
    "mni": "Manipuri  (Bengali/Meetei)",
    "mr":  "Marathi   (Devanagari)",
    "ne":  "Nepali    (Devanagari)",
    "or":  "Odia      (Odia)",
    "pa":  "Punjabi   (Gurmukhi)",
    "sa":  "Sanskrit  (Devanagari)",
    "sat": "Santali   (Ol Chiki)",
    "sd":  "Sindhi    (Perso-Arabic)",
    "ta":  "Tamil     (Tamil)",
    "te":  "Telugu    (Telugu)",
    "ur":  "Urdu      (Perso-Arabic)",
}

# which .txt file (without extension) holds each language's text.
# Hindi ships as 3 shards (hi-1.txt, hi-2.txt, hi-3.txt); we just use the
# first shard since we only need 1000 sentences.
FILE = {c: ("hi-1" if c == "hi" else c) for c in LANGS}

BASE_URL = "https://huggingface.co/datasets/ai4bharat/IndicCorpV2/resolve/main/data/"

N_PER_LANG = 1000
print(len(LANGS), "languages")
assert len(LANGS) == 23


23 languages


In [ ]:

SENT_END = re.compile(r"[।॥.!?؟]+")

def split_into_sentences(paragraph):
    '''Very simple rule based sentence tokenizer (no external library).'''
    if not paragraph:
        return []
    # cut wherever a sentence-ending mark occurs
    raw_pieces = SENT_END.split(paragraph)
    sentences = []
    for piece in raw_pieces:
        piece = piece.strip()
        piece = re.sub(r"\s+", " ", piece)   # collapse multiple spaces/newlines
        if piece == "":
            continue
        n_words = len(piece.split(" "))
        # keep only "reasonable" sentences: not too short, not a giant leftover
        if 4 <= n_words <= 40 and 10 <= len(piece) <= 300:
            sentences.append(piece)
    return sentences


In [ ]:
def collect_sentences_for_language(code, n_needed=N_PER_LANG, max_lines_to_scan=20000):
    '''
    Streams the raw .txt file for one language directly from the repo
    (this bypasses the repo's custom split/config resolution, which can be
    flaky depending on your `datasets` library version) and keeps pulling
    sentences out of lines/paragraphs until we have n_needed unique ones.
    '''
    url = BASE_URL + FILE[code] + ".txt"

    # the generic "text" loader treats every line in the file as one example
    ds = load_dataset("text", data_files=url, split="train", streaming=True)

    collected = []
    seen = set()
    lines_scanned = 0

    for example in ds:
        lines_scanned += 1
        paragraph = example.get("text", "")
        for sent in split_into_sentences(paragraph):
            if sent not in seen:
                seen.add(sent)
                collected.append(sent)
            if len(collected) >= n_needed:
                break
        if len(collected) >= n_needed or lines_scanned >= max_lines_to_scan:
            break

    if len(collected) < n_needed:
        print(f"  [WARNING] {code}: only found {len(collected)} sentences "
              f"(wanted {n_needed}). Using what we have.")

    return collected[:n_needed]


In [ ]:
# quick sanity check on one small file before committing to all 23 languages
test_ds = load_dataset("text", data_files=BASE_URL + FILE["dg"] + ".txt",
                        split="train", streaming=True)
print(next(iter(test_ds)))


{'text': 'पन्छानियां जंदियां हियां--'}


In [ ]:

all_rows = []   # list of dicts: {"label": ..., "sentence": ...}

for code in LANGS:
    print("Collecting:", code, "-", LANGS[code])
    sents = collect_sentences_for_language(code, N_PER_LANG)
    for s in sents:
        all_rows.append({"label": code, "sentence": s})

raw_df = pd.DataFrame(all_rows)
print(raw_df.shape)
raw_df.to_csv("indiccorp_lid_raw.csv", index=False)
raw_df.head()


Collecting: as - Assamese  (Bengali-Assamese)
Collecting: bd - Bodo      (Devanagari)
Collecting: bn - Bengali   (Bengali)
Collecting: dg - Dogri     (Devanagari)
Collecting: gom - Konkani   (Devanagari)
Collecting: gu - Gujarati  (Gujarati)
Collecting: hi - Hindi     (Devanagari)
Collecting: kha - Khasi     (Latin)
Collecting: kn - Kannada   (Kannada)
Collecting: ks - Kashmiri  (Perso-Arabic)
Collecting: mai - Maithili  (Devanagari)
Collecting: ml - Malayalam (Malayalam)
Collecting: mni - Manipuri  (Bengali/Meetei)
Collecting: mr - Marathi   (Devanagari)
Collecting: ne - Nepali    (Devanagari)
Collecting: or - Odia      (Odia)
Collecting: pa - Punjabi   (Gurmukhi)
Collecting: sa - Sanskrit  (Devanagari)
Collecting: sat - Santali   (Ol Chiki)
Collecting: sd - Sindhi    (Perso-Arabic)
Collecting: ta - Tamil     (Tamil)
Collecting: te - Telugu    (Telugu)
Collecting: ur - Urdu      (Perso-Arabic)
(23000, 2)


,label,sentence
0,as,ঘটনাৰ বিৱৰণি মতে বোকাকাতৰ কৈলাখাতত নিয়মিতভাৱে ...
1,as,বোকাখাতৰ কৈলাখাতত নিয়মিতভাৱে বহা এই জুৱাৰ থলীত...
2,as,ৰাণু মণ্ডলৰ সুৱদী কন্ঠত মুগ্ধ হৈ ভাইজান ছলমানে...
3,as,"বহুতে ইয়াকো কৈছে যে, ছলমানৰ আহিবলগীয়া ছবিৰ বাব..."
4,as,কিন্তু আঁচল কথাতো হল ছচিয়েল মিডিয়াত প্ৰচাৰিত এ...


## Step 2: Stratified 80 / 10 / 10 split

For every language we take its 1000 sentences, shuffle them, and cut:
- first 800  -> train
- next  100  -> validation
- last  100  -> test

Doing this separately per language and then combining is exactly what
"stratified sampling" means here: every language ends up with the same
800 / 100 / 100 counts in every split.


In [ ]:
train_rows, val_rows, test_rows = [], [], []

for lang, group in raw_df.groupby("label"):
    sentences = group["sentence"].tolist()
    random.shuffle(sentences)          # shuffle just this language's sentences

    n = len(sentences)
    n_train = int(0.8 * n)
    n_val   = int(0.1 * n)
    # whatever is left (handles rounding) goes to test
    train_part = sentences[:n_train]
    val_part   = sentences[n_train:n_train + n_val]
    test_part  = sentences[n_train + n_val:]

    for s in train_part:
        train_rows.append({"label": lang, "sentence": s})
    for s in val_part:
        val_rows.append({"label": lang, "sentence": s})
    for s in test_part:
        test_rows.append({"label": lang, "sentence": s})

train_df = pd.DataFrame(train_rows)
val_df   = pd.DataFrame(val_rows)
test_df  = pd.DataFrame(test_rows)

# shuffle the overall (multi-language) splits so languages are mixed up
train_df = train_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
val_df   = val_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
test_df  = test_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

# sanity check: every language should have the same count in each split
print(train_df["label"].value_counts())
print(val_df["label"].value_counts())
print(test_df["label"].value_counts())

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)


train: (18400, 2)
val:   (2300, 2)
test:  (2300, 2)
label
pa     800
bn     800
mai    800
as     800
ml     800
kha    800
sat    800
ne     800
gu     800
mr     800
ur     800
te     800
dg     800
hi     800
sd     800
ks     800
gom    800
kn     800
mni    800
ta     800
bd     800
sa     800
or     800
Name: count, dtype: int64
label
kha    100
sd     100
bn     100
or     100
gom    100
mai    100
sat    100
bd     100
ta     100
hi     100
pa     100
ur     100
ne     100
mni    100
kn     100
ml     100
gu     100
as     100
ks     100
te     100
mr     100
dg     100
sa     100
Name: count, dtype: int64
label
kha    100
sd     100
bn     100
or     100
gom    100
mai    100
sat    100
bd     100
ta     100
hi     100
pa     100
ur     100
ne     100
mni    100
kn     100
ml     100
gu     100
as     100
ks     100
te     100
mr     100
dg     100
sa     100
Name: count, dtype: int64


## Step 3: Build our OWN TF-IDF vectorizer

Features we use, per sentence:
- **word unigrams** (single whitespace-separated tokens)
- **word bigrams** (pairs of consecutive words)
- **character 2-grams, 3-grams, 4-grams** (computed per word, with `^`/`$`
  boundary markers so "start of word" / "end of word" information is kept)

TF-IDF formula (the classic, simple one):

```
tf(term, doc)  = count of term in doc
idf(term)      = ln( (1 + N) / (1 + df(term)) ) + 1        <- "smoothed" idf
tfidf(t, d)    = tf(t, d) * idf(t)
```
Then every document vector is **L2-normalized** (divided by its length) so
long and short sentences are comparable.

Because 23 languages x lots of n-grams can create an *enormous* vocabulary,
we keep only the `max_features` most frequent n-grams (this is a very common,
simple way to keep things manageable — a real product would use hashing or a
bigger machine).


In [ ]:
def get_word_ngram_features(sentence):
    '''word unigrams + word bigrams'''
    words = sentence.split(" ")
    words = [w for w in words if w != ""]
    feats = []
    for w in words:
        feats.append("W1_" + w)
    for i in range(len(words) - 1):
        feats.append("W2_" + words[i] + "_" + words[i + 1])
    return feats


def get_char_ngram_features(sentence, n):
    '''character n-grams computed per word, with boundary markers.'''
    words = sentence.split(" ")
    feats = []
    for w in words:
        if w == "":
            continue
        padded = "^" + w + "$"          # mark start/end of the word
        for i in range(len(padded) - n + 1):
            gram = padded[i:i + n]
            feats.append(f"C{n}_" + gram)
    return feats


def extract_all_features(sentence):
    '''Combine word (uni+bi) and char (2,3,4) n-gram features for ONE sentence.'''
    feats = []
    feats.extend(get_word_ngram_features(sentence))
    feats.extend(get_char_ngram_features(sentence, 2))
    feats.extend(get_char_ngram_features(sentence, 3))
    feats.extend(get_char_ngram_features(sentence, 4))
    return feats


In [ ]:
class SimpleTfidfVectorizer:
    '''
    A very plain, hand-written TF-IDF vectorizer.
    fit()        -> learns vocabulary + idf values from training sentences
    transform()  -> turns sentences into a dense numpy TF-IDF matrix
    '''

    def __init__(self, max_features=6000):
        self.max_features = max_features
        self.vocab = {}      # feature_string -> column index
        self.idf = None      # numpy array, idf[column index]

    def fit(self, sentences):
        # 1) document frequency: in how many sentences does each feature appear?
        df_counter = Counter()
        n_docs = len(sentences)

        for sent in sentences:
            feats_in_this_doc = set(extract_all_features(sent))  # set = count once per doc
            for f in feats_in_this_doc:
                df_counter[f] += 1

        # 2) keep only the most frequent `max_features` n-grams (student-level
        #    trick to keep the vocabulary a manageable size)
        most_common = df_counter.most_common(self.max_features)

        self.vocab = {}
        idf_values = []
        for idx, (feat, df) in enumerate(most_common):
            self.vocab[feat] = idx
            idf = math.log((1 + n_docs) / (1 + df)) + 1.0
            idf_values.append(idf)

        self.idf = np.array(idf_values, dtype=np.float32)
        print("Vocabulary size:", len(self.vocab))
        return self

    def transform(self, sentences):
        n_rows = len(sentences)
        n_cols = len(self.vocab)
        X = np.zeros((n_rows, n_cols), dtype=np.float32)

        for row, sent in enumerate(sentences):
            tf_counter = Counter(extract_all_features(sent))
            for feat, count in tf_counter.items():
                col = self.vocab.get(feat)
                if col is None:
                    continue   # unseen feature (not in training vocab) -> ignore
                X[row, col] = count * self.idf[col]

            # L2 normalize this row (this sentence's vector)
            norm = np.sqrt(np.sum(X[row] ** 2))
            if norm > 0:
                X[row] = X[row] / norm

        return X

    def fit_transform(self, sentences):
        self.fit(sentences)
        return self.transform(sentences)


In [ ]:
# Build the TF-IDF matrices.
# max_features keeps this runnable on a normal laptop; raise it if you have
# more RAM / time.
vectorizer = SimpleTfidfVectorizer(max_features=6000)

train_sentences = train_df["sentence"].tolist()
val_sentences   = val_df["sentence"].tolist()
test_sentences  = test_df["sentence"].tolist()

X_train = vectorizer.fit_transform(train_sentences)   # learn vocab ONLY from train
X_val   = vectorizer.transform(val_sentences)
X_test  = vectorizer.transform(test_sentences)

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)


Vocabulary size: 6000
X_train: (18400, 6000)
X_val:   (2300, 6000)
X_test:  (2300, 6000)


## Step 4: Our OWN multinomial Logistic Regression (softmax regression)

This is plain gradient-descent softmax regression:

```
scores  = X @ W + b                      # (n_samples, n_classes)
probs   = softmax(scores)                # row-wise softmax
loss    = -mean( log( probs[correct_class] ) )  + L2 penalty
gradients computed by hand, weights updated with:  W = W - lr * dW
```


In [ ]:
# Turn language-string labels into integer ids 0..22 (sorted so it's reproducible)
LABELS_SORTED = sorted(LANGS.keys())
label2id = {lab: i for i, lab in enumerate(LABELS_SORTED)}
id2label = {i: lab for lab, i in label2id.items()}
n_classes = len(LABELS_SORTED)
print("Number of classes:", n_classes)

y_train = np.array([label2id[l] for l in train_df["label"]])
y_val   = np.array([label2id[l] for l in val_df["label"]])
y_test  = np.array([label2id[l] for l in test_df["label"]])


def one_hot(y, n_classes):
    m = np.zeros((len(y), n_classes), dtype=np.float32)
    for i, cls in enumerate(y):
        m[i, cls] = 1.0
    return m


Y_train_oh = one_hot(y_train, n_classes)


Number of classes: 23


In [ ]:
class SoftmaxRegressionScratch:
    '''
    Plain multinomial logistic regression trained with mini-batch gradient
    descent. Written from scratch with numpy only (no sklearn).
    '''

    def __init__(self, n_features, n_classes, lr=0.5, epochs=60,
                 batch_size=256, l2=1e-4, seed=42):
        rng = np.random.default_rng(seed)
        # small random initial weights
        self.W = rng.normal(0, 0.01, size=(n_features, n_classes)).astype(np.float32)
        self.b = np.zeros((n_classes,), dtype=np.float32)
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2 = l2

    @staticmethod
    def softmax(scores):
        # subtract row max for numerical stability
        scores = scores - np.max(scores, axis=1, keepdims=True)
        exp_scores = np.exp(scores)
        return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

    def predict_proba(self, X):
        scores = X @ self.W + self.b
        return self.softmax(scores)

    def predict(self, X):
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)

    def fit(self, X, Y_onehot, X_val=None, y_val=None):
        n_samples = X.shape[0]

        for epoch in range(1, self.epochs + 1):
            # shuffle every epoch (simple, student-style mini-batching)
            perm = np.random.permutation(n_samples)
            X_shuf = X[perm]
            Y_shuf = Y_onehot[perm]

            for start in range(0, n_samples, self.batch_size):
                end = start + self.batch_size
                X_batch = X_shuf[start:end]
                Y_batch = Y_shuf[start:end]
                m = X_batch.shape[0]

                probs = self.predict_proba(X_batch)         # (m, n_classes)
                error = probs - Y_batch                      # (m, n_classes)

                grad_W = (X_batch.T @ error) / m + self.l2 * self.W
                grad_b = np.mean(error, axis=0)

                self.W -= self.lr * grad_W
                self.b -= self.lr * grad_b

            if epoch % 5 == 0 or epoch == 1 or epoch == self.epochs:
                train_probs = self.predict_proba(X)
                # cross entropy loss, small epsilon to avoid log(0)
                train_loss = -np.mean(np.sum(Y_onehot * np.log(train_probs + 1e-9), axis=1))
                train_acc = np.mean(np.argmax(train_probs, axis=1) == np.argmax(Y_onehot, axis=1))
                msg = f"epoch {epoch:3d} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f}"
                if X_val is not None and y_val is not None:
                    val_pred = self.predict(X_val)
                    val_acc = np.mean(val_pred == y_val)
                    msg += f" | val_acc {val_acc:.4f}"
                print(msg)

        return self


In [ ]:
model = SoftmaxRegressionScratch(
    n_features=X_train.shape[1],
    n_classes=n_classes,
    lr=0.5,
    epochs=500,
    batch_size=256,
    l2=1e-4,
)

model.fit(X_train, Y_train_oh, X_val=X_val, y_val=y_val)


epoch   1 | train_loss 2.8064 | train_acc 0.9040 | val_acc 0.8939
epoch   5 | train_loss 1.7669 | train_acc 0.9293 | val_acc 0.9139
epoch  10 | train_loss 1.1248 | train_acc 0.9360 | val_acc 0.9217
epoch  15 | train_loss 0.8405 | train_acc 0.9401 | val_acc 0.9309
epoch  20 | train_loss 0.6878 | train_acc 0.9437 | val_acc 0.9361
epoch  25 | train_loss 0.5933 | train_acc 0.9459 | val_acc 0.9357
epoch  30 | train_loss 0.5293 | train_acc 0.9470 | val_acc 0.9370
epoch  35 | train_loss 0.4831 | train_acc 0.9479 | val_acc 0.9374
epoch  40 | train_loss 0.4483 | train_acc 0.9486 | val_acc 0.9378
epoch  45 | train_loss 0.4212 | train_acc 0.9495 | val_acc 0.9383
epoch  50 | train_loss 0.3994 | train_acc 0.9502 | val_acc 0.9391
epoch  55 | train_loss 0.3815 | train_acc 0.9509 | val_acc 0.9391
epoch  60 | train_loss 0.3667 | train_acc 0.9515 | val_acc 0.9391
epoch  65 | train_loss 0.3541 | train_acc 0.9520 | val_acc 0.9387
epoch  70 | train_loss 0.3432 | train_acc 0.9523 | val_acc 0.9391
epoch  75 

## Step 5: Evaluation — macro-F1 written by hand (no sklearn.metrics)

For every class `c`:
```
TP = predicted c AND actually c
FP = predicted c AND actually NOT c
FN = actually c  AND predicted NOT c

precision_c = TP / (TP + FP)
recall_c    = TP / (TP + FN)
f1_c        = 2 * precision_c * recall_c / (precision_c + recall_c)
```
`macro-F1` = plain average of `f1_c` over all classes (every class counts
equally, no matter how many test sentences it has — which is fine here since
every language has the same number of test sentences anyway).


In [ ]:
    FdXqdef precision_recall_f1_per_class(y_true, y_pred, n_classes):
    results = {}
    for c in range(n_classes):
        tp = np.sum((y_pred == c) & (y_true == c))
        fp = np.sum((y_pred == c) & (y_true != c))
        fn = np.sum((y_pred != c) & (y_true == c))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        if precision + recall > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = 0.0

        results[c] = {"precision": precision, "recall": recall, "f1": f1,
                       "support": int(tp + fn)}
    return results


def macro_f1_score(y_true, y_pred, n_classes):
    per_class = precision_recall_f1_per_class(y_true, y_pred, n_classes)
    f1_values = [per_class[c]["f1"] for c in per_class]
    return sum(f1_values) / len(f1_values), per_class


In [ ]:
# --- Validation set report ---
val_pred = model.predict(X_val)
val_macro_f1, val_per_class = macro_f1_score(y_val, val_pred, n_classes)
print("VALIDATION macro-F1:", round(val_macro_f1, 4))

# --- Test set report (final numbers) ---
test_pred = model.predict(X_test)
test_macro_f1, test_per_class = macro_f1_score(y_test, test_pred, n_classes)
print("TEST macro-F1:", round(test_macro_f1, 4))
overall_acc = np.mean(test_pred == y_test)
print("TEST accuracy:", round(overall_acc, 4))


VALIDATION macro-F1: 0.9454
TEST macro-F1: 0.9423
TEST accuracy: 0.9422


In [ ]:

rows = []
for c in range(n_classes):
    stats = test_per_class[c]
    rows.append({
        "label": id2label[c],
        "precision": round(stats["precision"], 4),
        "recall": round(stats["recall"], 4),
        "f1": round(stats["f1"], 4),
        "support": stats["support"],
    })

report_df = pd.DataFrame(rows).sort_values("label").reset_index(drop=True)
print(report_df.to_string(index=False))
print()
print("MACRO-F1 (test):", round(test_macro_f1, 4))


label  precision  recall     f1  support
   as     1.0000    0.99 0.9950      100
   bd     0.9767    0.84 0.9032      100
   bn     0.9524    1.00 0.9756      100
   dg     0.9674    0.89 0.9271      100
  gom     0.6346    0.66 0.6471      100
   gu     1.0000    1.00 1.0000      100
   hi     0.7917    0.95 0.8636      100
  kha     0.8850    1.00 0.9390      100
   kn     0.9901    1.00 0.9950      100
   ks     1.0000    0.98 0.9899      100
  mai     0.9175    0.89 0.9036      100
   ml     1.0000    1.00 1.0000      100
  mni     1.0000    0.91 0.9529      100
   mr     0.7021    0.66 0.6804      100
   ne     0.9600    0.96 0.9600      100
   or     1.0000    1.00 1.0000      100
   pa     1.0000    1.00 1.0000      100
   sa     0.9604    0.97 0.9652      100
  sat     1.0000    0.98 0.9899      100
   sd     1.0000    1.00 1.0000      100
   ta     1.0000    1.00 1.0000      100
   te     1.0000    1.00 1.0000      100
   ur     0.9802    0.99 0.9851      100

MACRO-F1 (test)